# Speed Layer — Silver Transformation (Exploratory)
This notebook demonstrates exploratory steps to clean, validate, and write streaming events from the Bronze `events_bronze` table into a partitioned Delta `events_silver` table. Run on Databricks.

In [ ]:
# Imports and Spark session
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()
print('Spark session ready:', spark)

In [ ]:
# Preview bronze table (must exist in the catalog)
try:
    bronze = spark.table('events_bronze')
    print('Bronze count:', bronze.count())
    display(bronze.limit(20))
except Exception as e:
    print('Failed to read events_bronze table. Create or ingest bronze first.
', e)

In [ ]:
# Transform: validate fields, cast types, compute ingestion metadata
try:
    df = bronze
    # keep required fields only (example schema: timestamp, visitorid, event, itemid, transactionid)
    df = df.filter(F.col('timestamp').isNotNull() & F.col('visitorid').isNotNull() & F.col('event').isNotNull())
    df = df.withColumn('event_timestamp', F.col('timestamp').cast('long'))
    df = df.withColumn('item_id', F.col('itemid').cast('int'))
    df = df.withColumn('_ingestion_time', F.current_timestamp())
    df = df.withColumn('ingestion_latency_seconds', (F.col('_ingestion_time').cast('long') - (F.col('event_timestamp')/1000)).cast('double'))
    df = df.withColumn('event_date', F.to_date(F.from_unixtime(F.col('event_timestamp')/1000)))
    display(df.select('event_timestamp','visitorid','event','item_id','event_date','ingestion_latency_seconds').limit(20))
except NameError:
    print('`bronze` dataframe not available; run previous cell to load the table.')

In [ ]:
# Write the cleaned results to a Delta Silver table partitioned by event_date (exploratory append)
try:
    df.write.format('delta').mode('append').partitionBy('event_date').saveAsTable('events_silver')
    print('Wrote records to events_silver (append)')
except Exception as e:
    print('Write failed — ensure Delta and permissions available.
', e)

In [ ]:
# Quick validation queries on the newly written silver table
try:
    display(spark.sql('SELECT event, COUNT(*) as cnt FROM events_silver GROUP BY event ORDER BY cnt DESC'))
    display(spark.sql('SELECT MAX(_ingestion_time) as last_ingest FROM events_silver'))
except Exception as e:
    print('Validation queries failed — ensure events_silver exists.
', e)

---
Notes:
- This notebook is exploratory and intended for Databricks.
- For production, move transformation logic into `src/speed_layer/silver_layer/transform_events_silver.py` and call from job wrappers.